[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/service_assistant/openai_agents.ipynb)

# Simulated service assistant — OpenAI Agents SDK

A typed request Agent hands an exact proposal to deterministic approval and execution code, then a confirmation Agent reports the receipt. No external service or API key is used; mock runtime is under one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai-agents==0.17.8", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar

from agents import Agent, Runner, set_tracing_disabled
from agents.items import ModelResponse as SDKResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import ResponseOutputMessage, ResponseOutputText
from pydantic import BaseModel, ConfigDict

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture = json.loads((ROOT / "data/service_assistant/case_mock.json").read_text())

## Typed agents and exact-action gate
The SDK Agent proposes but cannot authorise or execute. Python checks least privilege and every approved argument, checkpoints, resumes and prevents duplicates by idempotency key.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: str
    order_id: str
    new_address: str
    idempotency_key: str
    requires_approval: bool


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: str


def core_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


class FixtureModel(SDKModel):
    def __init__(self):
        self.core = core_model()

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        response = await self.core.generate([user(str(input))])
        item = ResponseOutputMessage(
            id=response.response_id,
            content=[
                ResponseOutputText(
                    annotations=[],
                    text=json.dumps(response.structured_output, sort_keys=True),
                    type="output_text",
                    logprobs=[],
                )
            ],
            role="assistant",
            status="completed",
            type="message",
        )
        return SDKResponse(output=[item], usage=Usage(), response_id=response.response_id)

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


async def run_service():
    model = FixtureModel()
    service = SimulatedService.from_path(service_path)
    trace = [{"event": "read", "order": service.read_order("ord-100", actor="tutorial-user")}]
    request_agent = Agent(
        name="Request reviewer",
        instructions="Propose one typed action; never claim it executed.",
        model=model,
        output_type=Action,
    )
    action = (
        await Runner.run(request_agent, "Update the order address.", max_turns=2)
    ).final_output
    trace.append({"event": "proposal"})
    approved = {
        "order_id": "ord-100",
        "new_address": "2 Evidence Road",
        "idempotency_key": "address-ord-100-v1",
    }
    exact = approved == {
        "order_id": action.order_id,
        "new_address": action.new_address,
        "idempotency_key": action.idempotency_key,
    }
    authorised = action.action == "update_address" and action.requires_approval and exact
    trace.append({"event": "approval", "exact": exact})
    if not authorised:
        return {"outcome": "refused", "trace": trace}
    service = SimulatedService.resume(service.checkpoint())
    receipt = service.update_address(
        action.order_id,
        action.new_address,
        actor="tutorial-user",
        idempotency_key=action.idempotency_key,
    )
    duplicate = service.replay(action.idempotency_key)
    trace.extend([{"event": "effect"}, {"event": "duplicate_detected"}])
    confirmation_agent = Agent(
        name="Confirmer",
        instructions="Report only the supplied receipt.",
        model=model,
        output_type=Confirmation,
    )
    confirmation = (await Runner.run(confirmation_agent, str(receipt), max_turns=2)).final_output
    return {
        "action": action,
        "receipt": receipt,
        "duplicate": duplicate,
        "outcome": confirmation.status,
        "trace": trace,
        "model_calls": 2,
        "termination": "criteria_met",
    }


first = await run_service()
second = await run_service()
evaluation = {
    "component": first["receipt"]["delivery_address"] == "2 Evidence Road",
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 2,
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 2},
    "fallback": "non-exact approval refuses before effect",
}

## Limitation
The SDK handoff is in-process and simulated; production use requires durable approvals, authenticated principals and transactional service APIs.